# Load

Carga el dataset transformado, separa features y target, y genera los splits de entrenamiento y validación listos para modelar.

Input: `data_transformed.parquet`  
Output: `X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_parquet('../data/processed/data_transformed.parquet')
print('Shape:', df.shape)
df.head()

## 1. Verificación del dataset

In [2]:
print('Nulos:', df.isnull().sum().sum())
print('\nDistribución del target:')
print(df['target_churn_indicator'].value_counts())
print(f"\nChurn rate: {df['target_churn_indicator'].mean()*100:.1f}%")

Nulos: 0

Distribución del target:
target_churn_indicator
1    10348
0     8161
Name: count, dtype: int64

Churn rate: 55.9%


## 2. Separación de features y target

In [3]:
X = df.drop(columns=['user_id', 'target_churn_indicator'])
y = df['target_churn_indicator']

print('Features:', X.columns.tolist())
print('\nShape X:', X.shape)
print('Shape y:', y.shape)

Features: ['install_hour', 'install_dow', 'is_weekend', 'event_3_used', 'event_1_log', 'event_2_log', 'event_3_log', 'event_4_log', 'event_5_log', 'is_teen', 'platform_enc', 'gender_enc', 'region_enc']

Shape X: (18509, 13)
Shape y: (18509,)


## 3. Train / Test split

Split 80/20 con `stratify=y` para mantener la proporción de churn en ambos conjuntos.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train: {X_train.shape[0]} registros ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test:  {X_test.shape[0]} registros ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nChurn rate train: {y_train.mean()*100:.1f}%')
print(f'Churn rate test:  {y_test.mean()*100:.1f}%')

Train: 14807 registros (80%)
Test:  3702 registros (20%)

Churn rate train: 55.9%
Churn rate test:  55.9%


## 4. Export

In [ ]:
X_train.to_parquet('../data/splits/X_train.parquet', index=False)
X_test.to_parquet('../data/splits/X_test.parquet', index=False)
y_train.to_frame().to_parquet('../data/splits/y_train.parquet', index=False)
y_test.to_frame().to_parquet('../data/splits/y_test.parquet', index=False)

print('Archivos guardados en data/splits/')